# Forge F01 — Ücretsiz Colab GPU doğrulaması
Bu defter GPU ister; ücretli abonelik açmaz. Çalışma zamanı → Tümünü çalıştır. GPU yoksa veya ağ engelliyse durur. TripoSG yalnız deneysel resmi demo çıkarımıdır, F01 kabulü değildir. Model/ağırlık kullanım şartlarına uyulmalı; RMBG-1.4 ticari kullanım için ayrı lisans gerektirebilir.

In [ ]:
import sys, json, subprocess, pathlib, time, urllib.request
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü değiştir → T4 GPU → Kaydet."
gpu = torch.cuda.get_device_properties(0)
print("CUDA:", torch.cuda.is_available(), "| GPU:", gpu.name, "| VRAM GiB:", round(gpu.total_memory/2**30, 2))
assert gpu.total_memory >= 8 * 2**30, "TripoSG için en az 8 GiB VRAM gerekli."
x = torch.arange(1024, device="cuda", dtype=torch.float32)
assert float(x.sum().item()) == 523776.0
torch.cuda.synchronize()
print("GERÇEK CUDA HESABI GEÇTİ")
for url in ["https://github.com/VAST-AI-Research/TripoSG", "https://huggingface.co/api/models/VAST-AI/TripoSG", "https://huggingface.co/api/models/briaai/RMBG-1.4"]:
    with urllib.request.urlopen(url, timeout=25) as response:
        print(response.status, url)
print("Python:",sys.version,"PyTorch:",torch.__version__,"CUDA build:",torch.version.cuda)


## Resmi TripoSG demo testi
Bu hücre yeni ve deneysel kurulumdur. Colab Python/CUDA sürümü nedeniyle kurulum başarısız olabilir. Eski NumPy 1.22.3 sabiti modern Python ile uyumsuz olduğu için Python 3.10–3.12'de 1.26.4 kullanılır. Üretim motoru ya da ticari dağıtım olarak kullanılmaz.

In [ ]:
import os, sys, subprocess, pathlib, torch
assert torch.cuda.is_available()
assert (3,10) <= sys.version_info[:2] <= (3,12), "Bu deney Python 3.10–3.12 gerektiriyor; eski Colab çalışma zamanı seçilmesi gerekebilir."
root=pathlib.Path("/content/forge-f01-triposg")
repo="https://github.com/VAST-AI-Research/TripoSG.git"
rev="fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c"
def run(args, cwd=None, timeout=1200):
    subprocess.run(args,cwd=cwd,check=True,timeout=timeout)
if not root.exists():
    run(["git","clone",repo,str(root)])
run(["git","-C",str(root),"checkout","--detach",rev])
assert subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()==rev
run(["nvcc","--version"])
requirements=(root/"requirements.txt").read_text().splitlines()
deps=[l.strip() for l in requirements if l.strip() and not l.strip().startswith("#") and l.strip().split("#")[0].strip() not in ("diso","numpy==1.22.3")]
req=root/"forge-colab-requirements.txt"
req.write_text("\n".join(deps+["numpy==1.26.4","ninja","setuptools","wheel"])+"\n")
run([sys.executable,"-m","pip","install","-r",str(req)])
run([sys.executable,"-m","pip","install","--no-build-isolation","diso"])
run([sys.executable,"-c","import torch, diso, pymeshlab; assert torch.cuda.is_available(); print(torch.__version__)"],cwd=root)
print("Kurulum adımları tamamlandı; çıkarım henüz çalışmadı.")


In [ ]:
import hashlib, json, time, pathlib, subprocess, sys, torch, shutil
assert torch.cuda.is_available()
root=pathlib.Path("/content/forge-f01-triposg")
out=pathlib.Path("/content/forge-f01-gpu-evidence")
out.mkdir(exist_ok=True)
src=root/"assets/example_data/hjswed.png"
assert src.is_file()
dest=out/"actual_triposg_demo.glb"
if dest.exists(): dest.unlink()
t=time.monotonic()
subprocess.run([sys.executable,"-m","scripts.inference_triposg","--image-input",str(src),"--output-path",str(dest),"--seed","42"],cwd=root,check=True,timeout=1800)
import trimesh, numpy as np
m=trimesh.load(dest,force="mesh",process=False)
assert len(m.faces)>100 and np.isfinite(m.vertices).all() and np.all(m.extents>0)
sha=lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
shutil.copy2(src,out/"source.png")
report={"engine":"TripoSG official demo","actual_inference":True,"source_commit":rev,"gpu":torch.cuda.get_device_name(0),"source_sha256":sha(src),"glb_sha256":sha(dest),"faces":len(m.faces),"vertices":len(m.vertices),"watertight":bool(m.is_watertight),"seconds":time.monotonic()-t,"visual_accepted":False,"editor_accepted":False,"benchmark_10_completed":False}
(out/"report.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
print("Gerçek demo GLB üretildi; kalite, doku ve mobil bütçe ayrı doğrulanacak.")
archive=shutil.make_archive("/content/Forge_F01_GPU_Evidence","zip",out)
from google.colab import files
files.download(archive)
